## Convert Tif into PNG

In [ ]:
from pathlib import Path
from PIL import Image

in_dir = Path("./images")
out_dir = Path("./training_png")
out_dir.mkdir(parents=True, exist_ok=True)

for i, tif_path in enumerate(sorted(in_dir.glob("*.tif"))):
    img = Image.open(tif_path).convert("RGB")
    # resize if needed; many diffusion models use 512x512 or 768x768
    img = img.resize((512, 512))
    img.save(out_dir / f"img_{i:05d}.png")


## Generate txt labels for each png image based on csv file label

In [ ]:
# use number to represent the aggregate size
import pandas as pd

df = pd.read_csv("./images/tags_training.csv")
imgids, aggs, expansions = df["New_index"].values.tolist(), df["agg"].values.tolist(),df["expansion"].values.tolist()
print(len(imgids))
n_train = int(len(imgids)*0.8)
print(n_train)

for i in range(len(imgids)):
    caption = f"concrete microstructure image, aggregate size {aggs[i]}, expansion level {expansions[i]}"
    txt_path = f"./data/img_{imgids[i]:05d}.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(caption + "\n")

1596
1276


In [2]:
# use text to represent the aggregate size

import pandas as pd

df = pd.read_csv("./images/tags_training.csv")
imgids, aggs, expansions = df["New_index"].values.tolist(), df["agg"].values.tolist(),df["expansion"].values.tolist()
print(len(imgids))
n_train = int(len(imgids)*0.8)
print(n_train)

for i in range(len(imgids)):
    if aggs[i] == 4: aggsize = "large"
    elif aggs[i] == 3: aggsize = "medium"
    elif aggs[i] == 2: aggsize = "small"
    elif aggs[i] == 1: aggsize = "tiny"
    caption = f"aggregate size {aggsize}, expansion level {expansions[i]}"
    txt_path = f"./data/img_{imgids[i]:05d}.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(caption + "\n")

1596
1276


## Finally generate json format label file

In [3]:
import os, json

data_dir = "/home/wangz/project/genai/data"  # same as --train_data_dir

exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

files = [f for f in os.listdir(data_dir) if f.lower().endswith(exts)]
files.sort()

meta_path = os.path.join(data_dir, "metadata.jsonl")
with open(meta_path, "w", encoding="utf-8") as f:
    for fn in files:
        label_path = "./data/"+fn.replace(".png",".txt")
        with open(label_path,"r") as ftxt:
            caption = ftxt.readline()
            caption = caption.rstrip()
        rec = {
            "file_name": fn,   # relative path from data_dir
            "text": caption,   # this becomes the caption column
        }
        f.write(json.dumps(rec) + "\n")

print(f"Wrote {len(files)} entries to {meta_path}")


Wrote 1550 entries to /home/wangz/project/genai/data/metadata.jsonl
